In [0]:
%run ../00-common/01.environment-config

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
control_table=f"{catalog_name}.{control_schema}.batch_control"

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import lit
from delta.tables import DeltaTable

if v_batch_id:
    delta_table=DeltaTable.forName(spark,control_table)
    
    source_df=(
        spark.createDataFrame([(v_batch_id,)],["batch_id"])
        .withColumn("status",lit("completed"))
        .withColumn("updated_timestamp",F.current_timestamp())
    )
    
    (
        delta_table.alias("t")
        .merge(source_df.alias("s"), "t.batch_id = s.batch_id and t.status = 'IN_PROGRESS'")
        .whenMatchedUpdate(set = {"status": "s.status", "updated_timestamp": "s.updated_timestamp"})
        .execute()
    )
    
   
    print(f"Marked batch {v_batch_id} as completed")
else:
    raise Exception ("batch id is missing")